In [1]:
import re
from typing import List, Dict

# Robust PII regexes for Sinhala/English
EMAIL_RE = re.compile(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b", re.UNICODE)
NIC_RE = re.compile(r"(?<!\d)(\d{9}[VvXx]|\d{12})(?!\d)")
PHONE_RE = re.compile(r"(?x)(?<!\d)((?:(?:\+94)[\s-]*\d{2,3}(?:[\s-]*\d){2,7})|(?:0\d{2,3}(?:[\s-]*\d){2,7}))(?!\d)", re.UNICODE)

def _clean_phone(s: str) -> str:
    """Removes spaces and dashes from phone numbers"""
    return re.sub(r"[\s\-]", "", s)

def detect_structured_pii(text: str) -> Dict[str, List[str]]:
    """Detects NIC, Email, and Phone using regex"""
    emails = EMAIL_RE.findall(text)
    nics = NIC_RE.findall(text)
    raw_phones = PHONE_RE.findall(text)
    phones = [_clean_phone(p[0] if isinstance(p, tuple) else p) for p in raw_phones]
    return {"emails": emails, "nics": nics, "phones": phones}

def extract_address_text(text: str) -> str:
    """Extracts the physical address string based on common patterns"""
    # English patterns (e.g., No 45, Galle Rd)
    en_pattern = r"(No\s?\d+[\w\s,]+(?:Rd|Road|Street|Lane|Ave|Avenue|Town|Colombo|Galle|Kandy|Jaffna)[\w\s,.]*)"
    # Sinhala patterns (e.g., ලිපිනය 400/B, මල් පාර)
    si_pattern = r"(ලිපිනය\s?[\w\s,/-]+(?:පාර|වත්ත|මාවත|නගරය|කොළඹ|මහනුවර|ගාල්ල)[\w\s,.]*)"
    
    en_match = re.search(en_pattern, text, re.IGNORECASE)
    si_match = re.search(si_pattern, text)
    
    if en_match: return en_match.group(0).strip()
    if si_match: return si_match.group(0).strip()
    return "Address detected by model, but specific string could not be parsed."

In [2]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, f1_score

# Load dataset from Kaggle path
DATASET_PATH = '/kaggle/input/datasets/sudda02/sinhala-english-pii-data/combined_dataset.csv'
df = pd.read_csv(DATASET_PATH)
df['text'] = df['text'].fillna('')

# Split data 80/20
X_train, X_test, y_train, y_test = train_test_split(
    df['text'], df['physical_address'], test_size=0.2, random_state=42
)

# Vectorization (Multilingual support)
vectorizer = TfidfVectorizer(ngram_range=(1, 2))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# Train Hybrid component (Logistic Regression)
model = LogisticRegression(class_weight='balanced')
model.fit(X_train_tfidf, y_train)

# Calculate Evaluation Metrics for Display
y_pred = model.predict(X_test_tfidf)
eval_metrics = {
    "Precision": precision_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "F1": f1_score(y_test, y_pred)
}

# Save components
joblib.dump(model, 'address_model.pkl')
joblib.dump(vectorizer, 'vectorizer.pkl')
print("Model trained and metrics calculated successfully.")

Model trained and metrics calculated successfully.


In [ ]:
def pii_detector_interface():
    # Load saved toolsDatabase entry: Email: info7995@gmail.com, Address: No 592, Main St, Kandy.
    model = joblib.load('address_model.pkl')
    vec = joblib.load('vectorizer.pkl')
    
    print("=== MULTILINGUAL PII DETECTOR INTERFACE ===")
    
    while True:
        user_input = input("\nEnter message to analyze (or 'exit'): ")
        if user_input.lower() == 'exit': break
        
        # 1. Regex Detection
        reg_res = detect_structured_pii(user_input)
        
        # 2. ML Address Detection
        tfidf_input = vec.transform([user_input])
        address_detected = model.predict(tfidf_input)[0]
        
        # 3. Formatted Output
        print("\n" + "="*50)
        print("DETECTED SENSITIVE DATA:")
        print(f" - NIC:     {reg_res['nics'] if reg_res['nics'] else 'None'}")
        print(f" - Emails:  {reg_res['emails'] if reg_res['emails'] else 'None'}")
        print(f" - Phones:  {reg_res['phones'] if reg_res['phones'] else 'None'}")
        
        if address_detected:
            print(f" - Address: {extract_address_text(user_input)}")
        else:
            print(" - Address: None Detected")
            
        print("-" * 50)
        print("MODEL EVALUATION METRICS (Hybrid Component 3):")
        print(f" Precision: {eval_metrics['Precision']:.2f}")
        print(f" Recall:    {eval_metrics['Recall']:.2f}")
        print(f" F1-Score:  {eval_metrics['F1']:.2f}")
        print("="*50)

# Run the interface
pii_detector_interface()

=== MULTILINGUAL PII DETECTOR INTERFACE ===



Enter message to analyze (or 'exit'):  Sensitive info: Phone: 077 7759125, Address: No 820, Galle Rd, Colombo, NIC: 621505042V.



DETECTED SENSITIVE DATA:
 - NIC:     ['621505042V']
 - Emails:  None
 - Phones:  ['0777759125']
 - Address: No 820, Galle Rd, Colombo, NIC
--------------------------------------------------
MODEL EVALUATION METRICS (Hybrid Component 3):
 Precision: 0.90
 Recall:    1.00
 F1-Score:  0.95
